[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/06_cdr_safe_tools/06_tools_lab.ipynb)


# 06 — CDR-safe 도구 — Humatch · AnthroAb · 3도구 합의

> 본문 [`06_cdr_safe_tools.md`](06_cdr_safe_tools.md) 와 **한 절씩 짝지어** 보세요.
> **실측 소요 —** Humatch 첫 실행 **160초**(Zenodo CNN 가중치 다운로드 포함) · AnthroAb `predict_best_score` **1.3~1.5초** / masked **2.3~2.5초**
> **앞 랩에서 이어져요** — Ch.04 numbering 의 `my_run/` 을 먼저 찾고, 없으면 커밋된 `data/` 로 대신합니다.

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `06_cdr_safe_tools/` 로 이동해 필요한 패키지만 깝니다(로컬에서 `06_cdr_safe_tools/` 안에 열었다면 클론 생략).
끝나면 **`my_run/`**(내가 만들 결과)과 **`data/`**(커밋된 레퍼런스)가 준비돼요 — 랩은 `my_run/` 을 먼저 찾고 없으면 `data/` 로 폴백하며 어느 쪽을 썼는지 매번 print 합니다.
- ANARCI 는 numbering 을 **hmmscan(HMMER)** 서브프로세스로 돌려요. Colab 에서는 `apt-get install -y hmmer` 가 함께 실행돼요 — pip 만으로는 `hmmscan` 이 없어 죽어요.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "06_cdr_safe_tools"
APT_PKGS = "hmmer"     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib anarci abnumber"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# 라이브러리 내부 소음 끄기 — 아래 두 줄은 결과와 무관한데 셀 출력 첫머리를 덮어요.
#   · tqdm "IProgress not found"  : 진행바를 위젯 대신 글자로 그린다는 안내일 뿐
#   · pkg_resources is deprecated : lightning/torch 가 남긴 setuptools 경고
# 진단이 필요하면 이 두 줄을 지우고 다시 실행하세요.
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")
warnings.filterwarnings("ignore", message=".*Bio.pairwise2 has been deprecated.*")
# 가중치 로딩 진행바와 transformers 안내문도 결과를 밀어내요 (import 전에 꺼 둡니다)
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)   # "unauthenticated requests" 안내

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True, timeout=None, quiet=False):
    # timeout 을 꼭 주세요 — 네트워크가 '멈춘 채' 매달리면 예외가 안 나서 data/ 폴백이 안 걸립니다.
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        return subprocess.run(cmd, shell=True, check=check, timeout=timeout)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        if check:
            raise subprocess.CalledProcessError(p.returncode, cmd)
    return p

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

_APT = APT_PKGS.split() if (APT_PKGS and IN_COLAB) else []   # 모아서 apt 를 한 번만 돌립니다
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")             # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨져요

if _APT:                                   # apt 인덱스 갱신은 한 번만 (ANARCI 는 hmmscan 이 필요해요)
    _run("apt-get -qq update", timeout=600, quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), timeout=900, quiet=True)


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — Humatch 설치 + humanization (본문 6.1.1~6.1.2)

Ch.05 의 Sapiens 는 가드 없이 돌리자 CDR 을 건드렸어요. Humatch 는 그 걱정을 도구 안으로 가져간 쪽이에요 —
**CDR 보호가 기본값**(`allow_CDR_mutations=False`)이고, framework 만 CNN 점수 목표에 닿을 때까지 single-point 로 반복 탐색해요.

`pip install humatch` 가 안 되면 **GitHub source** 로 자동 전환해요(본문 6.1.1 의 첫 번째 함정).
첫 실행은 Zenodo 에서 CNN 가중치(heavy/light/paired)와 germline 룩업 배열을 받으므로 **160초** 걸려요(실측).
다운로드가 멈추면 예외가 안 나서 폴백도 안 걸리니, **600초 타임아웃**을 걸어 멈춤을 실패로 바꿔 둡니다.

In [ ]:
import pandas as pd

# Humatch 는 파이썬 모듈명이 `Humatch`(대문자) 라서 소문자 import 체크가 통하지 않아요 → CLI 존재로 판정
if not shutil.which("Humatch-humanise"):
    _run(f'"{sys.executable}" -m pip -q install humatch', check=False)
if not shutil.which("Humatch-humanise"):
    print("pip 실패 → GitHub source 로 재시도해요")
    _run(f'"{sys.executable}" -m pip -q install git+https://github.com/oxpig/Humatch.git', check=False)
print("Humatch CLI:", shutil.which("Humatch-humanise") or "없음")

# CLI 는 CSV 입력/CSV 출력이 가장 안전해요(문자열 인자 -H/-L 도 가능).
inp = MY / "humatch_in.csv"
inp.write_text("VH,VL\n%s,%s\n" % (VH, VL))
out = MY / "humatch_out.csv"
cmd = ["Humatch-humanise", "-i", str(inp), "--vh_col", "VH", "--vl_col", "VL", "-o", str(out), "-v"]

hm_h = hm_l = hm_row = None
env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL="3")
t0 = time.time()
try:
    r = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=600)
    if r.returncode != 0 and "DNN library" in (r.stdout + r.stderr):
        # TensorFlow 가 GPU cuDNN 초기화에 실패하는 환경이 있어요 → CPU 로 강제하고 재시도(실측 사례)
        print("TensorFlow GPU 초기화 실패 → CPU 로 재시도해요")
        env["CUDA_VISIBLE_DEVICES"] = ""
        r = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=600)
    if r.returncode == 0 and out.exists():
        print(f"Humatch 완료 {time.time()-t0:.1f}초 (첫 실행은 Zenodo 가중치 다운로드 포함 — 실측 160초)")
        hm_row = pd.read_csv(out).iloc[0]
        hm_h = hm_row["Humatch_H"].replace("-", "")     # 출력은 200-position 정렬형 → gap 제거
        hm_l = hm_row["Humatch_L"].replace("-", "")
        write_fasta(MY / "humatch_humanised.fasta",
                    {"humatch_humanised_VH": hm_h, "humatch_humanised_VL": hm_l})
    else:
        print("Humatch 실행 실패:", (r.stdout + r.stderr).strip()[-300:])
except subprocess.TimeoutExpired:
    print("Humatch 600초 초과 — Zenodo 다운로드가 멈춘 것 같아요. 셀을 다시 실행하면 받다 만 곳부터 이어받아요.")
except Exception as e:
    print("Humatch 실행 실패:", type(e).__name__, str(e)[:200])

if hm_h is None or hm_l is None:
    ref = read_fasta(REF / "humatch_humanized.fasta")
    hm_h, hm_l = ref["humatch_humanised_VH"], ref["humatch_humanised_VL"]
    print("[레퍼런스]", REF / "humatch_humanized.fasta")

print(f"parental VH {len(VH)} aa · VL {len(VL)} aa  →  Humatch VH {len(hm_h)} aa · VL {len(hm_l)} aa")

## 2) 내 결과 확인 — config 가 약속한 것과 결과가 맞나 (본문 6.1.3~6.1.4)

Humatch 는 실행 로그에 **자기가 무엇을 지킬지 config 로 먼저 선언**해요. 그 선언과 실제 산출물을 맞춰 보는 게 이 절의 일이에요.

그런데 raw 인덱스로 비교하면 안 돼요 — Humatch 는 우리 예제의 **VL 에서 1 잔기를 지워요**(111 → 110 aa).
그래서 CDR 보존 확인은 **"parental CDR 문자열이 후보 안에 그대로 있는가"** 로 합니다(indel 에 안전한 방법).

In [ ]:
cfg = pd.read_csv(find_one("humatch_scores.csv")).set_index("metric")["value"]
print("   ↑ 이 CSV 가 [레퍼런스] 로 뜨는 건 정상이에요 — Humatch CLI 는 요약 CSV 를 만들지 않아요.")
print("     여기서 읽는 건 Humatch 의 config 기본값뿐이고, 점수·gene·edit 은 아래에서 내 실행 결과로 덮어씁니다.")

ct = pd.read_csv(find_prev("04_sequence_qc", "cdr_table_imgt.csv", quiet=True))
cdrs = {(r["chain"], r["cdr"]): r["sequence"] for _, r in ct.iterrows()}

if hm_row is not None:
    cnn_h, cnn_l, cnn_p = float(hm_row["CNN_H"]), float(hm_row["CNN_L"]), float(hm_row["CNN_P"])
    hv, lv, edit = hm_row["HV"], hm_row["LV"], int(hm_row["Edit"])
    SRC_ROW = "내 실행 (my_run/humatch_out.csv)"
else:
    cnn_h, cnn_l, cnn_p = float(cfg["CNN_H"]), float(cfg["CNN_L"]), float(cfg["CNN_P"])
    hv, lv, edit = cfg["HV_gene"], cfg["LV_gene"], int(cfg["edit_total"])
    SRC_ROW = "레퍼런스 (1절이 실패해 커밋본으로 이어감)"

# CDR 변이 수는 config 가 아니라 '결과' 예요 — 커밋본 숫자를 믿지 않고 내 산출물에서 직접 셉니다.
cdr_obs = sum(1 for (chain, _), s in cdrs.items()
              if s not in (hm_h if chain == "H" else hm_l))

tgt = float(cfg["CNN_target_score"])
display(pd.DataFrame([
    {"항목": "allow_CDR_mutations", "값": str(cfg["allow_CDR_mutations"]),
     "출처": "Humatch 기본값", "뜻": "CDR 은 기본 보호"},
    {"항목": "GL_target_score", "값": str(cfg["GL_target_score"]),
     "출처": "Humatch 기본값", "뜻": "germline-likeness 매칭 목표"},
    {"항목": "CNN_target_score", "값": str(cfg["CNN_target_score"]),
     "출처": "Humatch 기본값", "뜻": "이 점수에 닿을 때까지 framework 탐색"},
    {"항목": "CDR 변이 (실측)", "값": f"{cdr_obs}개",
     "출처": "내 산출물에서 직접 셈", "뜻": "위 선언이 실제로 지켜졌는지"},
]))

print(f"\nHumatch 가 낸 점수 — 출처 {SRC_ROW}")
display(pd.DataFrame([
    {"항목": "germline HV / LV", "값": f"{hv} / {lv}", "목표 대비": "-"},
    {"항목": "edit — 바꾼 자리 수", "값": str(edit), "목표 대비": "-"},
    {"항목": "CNN heavy", "값": f"{cnn_h:.3f}",
     "목표 대비": ("도달" if cnn_h >= tgt else "미달") + f" (목표 {tgt:g})"},
    {"항목": "CNN light", "값": f"{cnn_l:.3f}",
     "목표 대비": ("도달" if cnn_l >= tgt else "미달") + f" (목표 {tgt:g})"},
    {"항목": "CNN paired", "값": f"{cnn_p:.3f}", "목표 대비": "-"},
]))

def cdr_intact(cand_h, cand_l, label):
    rows = []
    for (chain, name), s in cdrs.items():
        seq = cand_h if chain == "H" else cand_l
        rows.append({"후보": label, "CDR": f"{chain}-{name}", "parental CDR": s, "보존": s in seq})
    return rows

chk = pd.DataFrame(cdr_intact(hm_h, hm_l, "Humatch"))
display(chk)
print(f"→ CDR 보존 {int(chk['보존'].sum())} / {len(chk)}.  VL 길이 {len(VL)} → {len(hm_l)}(1 잔기 삭제 = indel).")
print("   config 가 선언한 CDR 보호가 산출물에서도 지켜졌는지가 판정 기준이에요 — 6/6 이면 통과.")

## 3) 직접 실행 — AnthroAb 자동 모드 (본문 6.2.1)

| 모드 | 함수 | 무엇을 바꾸나 |
|---|---|---|
| ① 자동 전체 변경 | `predict_best_score(seq, chain)` | **모든 position** 을 가장 사람다운 잔기로 — **CDR 도 바꿔요** |
| ② 커스텀 마스킹 | `predict_masked(seq, chain)` | 내가 `*` 로 찍은 자리만 |

①을 그대로 돌려 봐요(실측 VH 1.5초 / VL 1.3초). 같은 CDR 보존 검사를 그대로 통과시켜 Humatch 와 나란히 놓습니다.

In [ ]:
_ensure("anthroab")

def mutations(par, hum):
    return [{"position_1based": i + 1, "wt": a, "mut": b, "mutation": f"{a}{i+1}{b}"}
            for i, (a, b) in enumerate(zip(par, hum)) if a != b]

ab_h = ab_l = None
try:
    import anthroab
    t0 = time.time()
    ab_h = anthroab.predict_best_score(VH, "H")
    ab_l = anthroab.predict_best_score(VL, "L")
    print(f"predict_best_score VH+VL {time.time()-t0:.1f}초 (실측 1.3~1.5초/체인)")
    write_fasta(MY / "anthroab_best_score.fasta",
                {"anthroab_bestscore_VH": ab_h, "anthroab_bestscore_VL": ab_l})
except Exception as e:
    print("AnthroAb 실행 실패:", type(e).__name__, str(e)[:160])

if ab_h is None or ab_l is None:
    f = read_fasta(REF / "anthroab_best_score.fasta")
    ab_h, ab_l = f["anthroab_predict_best_score_VH"], f["anthroab_predict_best_score_VL"]
    print("[레퍼런스]", REF / "anthroab_best_score.fasta")

ab_mut_h = pd.DataFrame(mutations(VH, ab_h)); ab_mut_l = pd.DataFrame(mutations(VL, ab_l))
chk2 = pd.DataFrame(cdr_intact(ab_h, ab_l, "AnthroAb(best_score)"))
display(chk2)
print(f"VH {len(ab_mut_h)} mut · VL {len(ab_mut_l)} mut · CDR 보존 {int(chk2['보존'].sum())} / {len(chk2)}")
broken = [r["CDR"] for _, r in chk2.iterrows() if not r["보존"]]
print("→ 파손된 CDR:", ", ".join(broken) if broken else "없음",
      "— 자동 모드는 CDR 을 지켜 주지 않아요(본문 6.2.1 경고 그대로).")

## 4) 직접 실행 — `predict_masked` 의 버그와 우회 (실측 발견)

**anthroab 1.1.0 의 `predict_masked()` 는 그대로 쓰면 안 됩니다.** docstring 은 `*`/`X` 로 마스킹하라고 하지만,
`hemantn/roberta-base-humAb-*` 의 tokenizer vocab 에는 `*`·`X` 가 **없어요**. 사전에 없는 문자는 `<unk>` 도 아니고 **조용히 삭제**되어
그 뒤 토큰이 한 칸씩 밀려요. 라이브러리는 (원래 길이의) 입력과 (짧아진) 예측을 zip 해서 **말없이 어긋난 결과**를 내거나 `IndexError` 로 죽어요.

우회는 간단해요 — 문자열 마스킹을 건너뛰고 **`input_ids` 를 직접 만들어** 진짜 `mask_token_id` 를 꽂습니다.

여기서 마스킹 대상은 **FR 전체**예요. 본문 6.2.5 의 masking 등급표(FWR exposed ✅ / buried ⚠️ / Vernier 🔸 / interface ⚠️ / CDR ⛔)를
**의도적으로 무시한 최대 마스킹**이고, CDR 보호만 지킨 상한선 실험이에요. 실무 기본값은 등급표대로 **FWR exposed 만** 찍는 쪽입니다.

In [ ]:
_ensure("torch transformers")

MODELS = {"H": "hemantn/roberta-base-humAb-vh", "L": "hemantn/roberta-base-humAb-vl"}
_LOADED = {}

def _load(chain):
    """체인당 한 번만 로드(가중치 164MB — 매 호출 로드하면 그만큼 매번 기다려요)."""
    if chain not in _LOADED:
        from transformers import AutoModelForMaskedLM, AutoTokenizer
        tok = AutoTokenizer.from_pretrained(MODELS[chain])
        mdl = AutoModelForMaskedLM.from_pretrained(MODELS[chain]); mdl.eval()
        _LOADED[chain] = (tok, mdl)
    return _LOADED[chain]

def masked_fill(seq, chain, mask_positions):
    """mask_positions(1-based)만 마스킹해 예측 — tokenizer 문자열 마스킹을 우회한 정공법.
    (import 를 함수 안에 둬요 — torch/transformers 가 어긋난 환경에서도 셀이 통째로 죽지 않게)"""
    import torch
    tok, mdl = _load(chain)
    vocab = tok.get_vocab()
    ids = [tok.bos_token_id] + [vocab[a] for a in seq] + [tok.eos_token_id]
    x = torch.tensor(ids).unsqueeze(0)
    for p in mask_positions:
        x[0, p] = tok.mask_token_id                 # +1 offset (bos) 이 이미 반영된 인덱스
    with torch.no_grad():
        logits = mdl(input_ids=x).logits[0]
    inv = {v: k for k, v in vocab.items()}
    outs = list(seq)
    for p in mask_positions:
        outs[p - 1] = inv[int(logits[p].argmax())]
    return "".join(outs)

# CDR 보호 좌표(Ch.04) → FR 자리만 마스킹
gp = GUIDE_ROOT / "04_sequence_qc" / "my_run" / "cdr_guard.json"
if gp.exists():
    guard = json.loads(gp.read_text()); print(f"[내 결과 · 04_sequence_qc] {gp}")
else:
    guard = {"H": [], "L": []}
    for _, r in ct.iterrows():
        seq = VH if r["chain"] == "H" else VL
        st = seq.find(r["sequence"]) + 1
        guard[r["chain"]] += list(range(st, st + len(r["sequence"])))
    print("CDR 좌표를 CDR 표에서 복원했어요")

fr_h = [i for i in range(1, len(VH) + 1) if i not in guard["H"]]
fr_l = [i for i in range(1, len(VL) + 1) if i not in guard["L"]]
print(f"마스킹 비율 — VH {len(fr_h)}/{len(VH)} · VL {len(fr_l)}/{len(VL)} (CDR 은 0)")

mk_h = mk_l = None
try:
    t0 = time.time()
    mk_h = masked_fill(VH, "H", fr_h)
    mk_l = masked_fill(VL, "L", fr_l)
    print(f"predict_masked(우회 구현) VH+VL {time.time()-t0:.1f}초 — 실측 2.3~2.5초/체인")
    write_fasta(MY / "anthroab_masked_FRonly.fasta",
                {"anthroab_masked_VH": mk_h, "anthroab_masked_VL": mk_l})
except Exception as e:
    print("masked 실행 실패:", type(e).__name__, str(e)[:160])

if mk_h is None or mk_l is None:
    f = read_fasta(REF / "anthroab_masked_FRonly.fasta")
    mk_h, mk_l = f["anthroab_predict_masked_fixed_VH"], f["anthroab_predict_masked_fixed_VL"]
    print("[레퍼런스]", REF / "anthroab_masked_FRonly.fasta")

mk_mut_h = pd.DataFrame(mutations(VH, mk_h)); mk_mut_l = pd.DataFrame(mutations(VL, mk_l))
chk3 = pd.DataFrame(cdr_intact(mk_h, mk_l, "AnthroAb(masked FR-only)"))
print(f"FR-only masked → VH {len(mk_mut_h)} mut · VL {len(mk_mut_l)} mut · "
      f"CDR 보존 {int(chk3['보존'].sum())} / {len(chk3)} (마스킹하지 않았으니 구조적으로 보장)")

if "H" in _LOADED:   # 본문 6.3.2 — 덩치를 로드한 모델에서 그대로 확인
    _cfg, _mdl = _LOADED["H"][1].config, _LOADED["H"][1]
    print(f"AnthroAb VH 모델 — {_cfg.num_hidden_layers}층 · hidden {_cfg.hidden_size} · "
          f"파라미터 {sum(p.numel() for p in _mdl.parameters())/1e6:.1f}M")
    print("   Sapiens(4층 · hidden 128 · 약 0.5M) 와 같은 API 를 쓰지만 덩치가 다른 모델이에요.")

## 5) 직접 실행 — 3도구 합의 (본문 6.2.6)

여기가 이 챕터의 핵심이에요. **도구 간 비교는 반드시 IMGT 번호로** 합니다(Humatch 의 indel 때문에 raw 인덱스가 어긋나므로).
Ch.04 에서 만든 `raw2imgt_*.json` 을 그대로 씁니다.

세는 방법을 두 가지로 나눠요. **세 도구가 모두 건드린 위치**와, 그중 **똑같은 잔기로 바꾼 위치(동일 치환)** 는 다른 숫자예요.

In [ ]:
def load_map(tag):
    p = GUIDE_ROOT / "04_sequence_qc" / "my_run" / f"raw2imgt_{tag}.json"
    if p.exists():
        print(f"[내 결과 · 04_sequence_qc] {p}")
    else:
        p = REF / f"raw2imgt_{tag}.json"; print(f"[레퍼런스] {p}")
    return {int(k): v for k, v in json.loads(p.read_text()).items()}

r2i = {"H": load_map("H"), "L": load_map("L")}

def to_imgt(df, chain, tool):
    return pd.DataFrame([{"chain": chain, "imgt": r2i[chain][int(r.position_1based)],
                          "wt": r.wt, "mut": r.mut, "sub": f"{r.wt}{r.mut}", "tool": tool}
                         for r in df.itertuples()] or None)

sap_h = pd.read_csv(find_prev("05_humanize_sapiens", "mutations_VH.csv", quiet=True))
sap_l = pd.read_csv(find_prev("05_humanize_sapiens", "mutations_VL.csv", quiet=True))

# Humatch — 내 결과에서 IMGT 로 재계산. 단 indel 이 난 체인은 raw 인덱스가 통째로 밀려
# 자리별 비교 자체가 성립하지 않아요 → 그 체인만 커밋된 IMGT 표를 씁니다(어느 쪽인지 아래에 출력).
hm_parts, hm_src = [], {}
ref_hmt = pd.read_csv(REF / "humatch_mutations_imgt.csv")
for chain, par_seq, cand_seq in (("H", VH, hm_h), ("L", VL, hm_l)):
    if len(par_seq) == len(cand_seq):
        hm_parts.append(to_imgt(pd.DataFrame(mutations(par_seq, cand_seq)), chain, "Humatch"))
        hm_src[chain] = "내 결과에서 재계산"
    else:
        sub = ref_hmt[ref_hmt.chain == chain]
        hm_parts.append(pd.DataFrame([{"chain": chain, "imgt": r.imgt_position, "wt": r.wt, "mut": r.mut,
                                       "sub": f"{r.wt}{r.mut}", "tool": "Humatch"} for r in sub.itertuples()]))
        hm_src[chain] = f"레퍼런스 IMGT 표(indel {len(par_seq)}→{len(cand_seq)} aa 라 raw 비교 불가)"
print("Humatch mutation 출처 —", " · ".join(f"{k}: {v}" for k, v in hm_src.items()))

tbl = pd.concat([
    to_imgt(sap_h, "H", "Sapiens"), to_imgt(sap_l, "L", "Sapiens"),
    to_imgt(ab_mut_h, "H", "AnthroAb_best_score"), to_imgt(ab_mut_l, "L", "AnthroAb_best_score"),
    to_imgt(mk_mut_h, "H", "AnthroAb_masked"), to_imgt(mk_mut_l, "L", "AnthroAb_masked"),
] + hm_parts, ignore_index=True)
tbl.to_csv(MY / "mutations_by_tool_imgt.csv", index=False)

def consensus(anthroab_mode):
    keep = tbl[tbl.tool.isin(["Sapiens", "Humatch", anthroab_mode])]
    out = []
    for (chain, pos), sub in keep.groupby(["chain", "imgt"]):
        if len(set(sub.tool)) == 3:
            out.append({"chain": chain, "imgt": pos, "subs": "/".join(sorted(set(sub["sub"]))),
                        "동일 치환": len(set(sub["sub"])) == 1})
    return pd.DataFrame(out).sort_values(["chain", "imgt"])

cons, counts = {}, {}
for mode in ["AnthroAb_best_score", "AnthroAb_masked"]:
    cs = consensus(mode)
    ident = cs[cs["동일 치환"]]
    cons[mode], counts[mode] = cs, (len(cs), len(ident))
    cs.to_csv(MY / f"three_way_consensus_{mode}.csv", index=False)
    print(f"\n=== Sapiens · Humatch · {mode} ===")
    print(f"세 도구가 모두 건드린 위치 {len(cs)}개 · 그중 동일 치환 {len(ident)}개")
    display(cs)
    split = cs[~cs["동일 치환"]]
    if len(split):
        print("   갈린 자리:", ", ".join(f"{r.imgt} {r.subs}" for r in split.itertuples()))

## 6) 내 결과 확인 — 합의 개수는 **어느 모드로 비교했느냐**에 달려 있어요

같은 세 도구인데 AnthroAb 를 `best_score` 로 놓느냐 `masked` 로 놓느냐에 따라 합의 목록이 바뀌어요.
그래서 합의를 인용할 때는 **개수만이 아니라 모드까지** 함께 적어야 해요.

지도는 `masked` 모드 기준으로 그려요. 금색 배경 + 빨간 테두리가 **세 도구 동일 치환**입니다.

In [ ]:
from humanization_viz import mutation_map
from IPython.display import Image, display

vh = tbl[(tbl.chain == "H") & (tbl.tool.isin(["Sapiens", "Humatch", "AnthroAb_masked"]))].copy()
vh = vh[vh["imgt"].str[1:].str.isdigit()]           # 'L127_tail1' 처럼 IMGT 범위 밖 라벨은 제외
vh["position"] = vh["imgt"].str[1:].astype(int)
pos3 = sorted(p for p, n in vh.groupby("position")["tool"].nunique().items() if n == 3)
rows = [{"position": r.position, "tool": r.tool, "to": r.mut}
        for r in vh.itertuples() if r.position in pos3]

# CDR 보호 구간도 데이터에서 — CDR 표(raw) → IMGT 경계로 변환
cdr_spans = []
for _, r in ct[ct.chain == "H"].iterrows():
    st = VH.find(r["sequence"]) + 1
    lo, hi = r2i["H"][st], r2i["H"][st + len(r["sequence"]) - 1]
    cdr_spans.append((int(lo[1:]), int(hi[1:])))
in_cdr = [p for p in pos3 if any(lo <= p <= hi for lo, hi in cdr_spans)]

png = mutation_map(rows, "3-tool proposals on VH (IMGT) — 금색 = 세 도구 동일 치환",
                   MY / "06_mutation_map.png",
                   cdr_spans=cdr_spans if in_cdr else None)   # 그릴 게 없으면 범례도 띄우지 않아요
print("CDR(IMGT) 구간:", ", ".join(f"H{lo}~H{hi}" for lo, hi in cdr_spans))
print(f"세 도구가 함께 건드린 VH 위치 {len(pos3)}개 중 CDR 안 {len(in_cdr)}개 — 합의는 전부 framework 에서 났어요.")
display(Image(str(png)))

h86 = vh[vh.position == 86][["tool", "wt", "mut"]]
n_touch, n_same = counts["AnthroAb_masked"]
if len(h86):
    print("\nIMGT H86 (= raw I78T) 에 대한 세 도구의 제안")
    display(h86)
    agreed = len(h86) == 3 and len(set(h86["mut"])) == 1
    print(f"판정 — masked 모드 합의 {n_same}/{n_touch}. H86 은 "
          + (f"세 도구가 모두 {h86.iloc[0]['wt']}→{h86.iloc[0]['mut']} 로 모인 자리라 backmutation 우선순위에서 가장 뒤로 밀 수 있어요."
             if agreed else "세 도구가 갈린 자리라 합의로 세지 않아요."))
else:
    print(f"\n판정 — masked 모드 합의 {n_same}/{n_touch}. 이 서열에서는 H86 을 세 도구가 함께 건드리지 않았어요.")
split = cons["AnthroAb_masked"]
split = split[~split["동일 치환"]]
if len(split):
    print("        제안 잔기가 갈린 자리(" + ", ".join(split["imgt"]) + ")는 합의로 세지 않아요.")

## 7) 레퍼런스 대조

In [ ]:
# keep_default_na=False — 치환 표기 'NA'(Asn→Ala)가 결측(NaN)으로 읽히는 것을 막아요
ref_cs = pd.read_csv(REF / "three_way_consensus.csv", keep_default_na=False)
MODE_KEY = {"AnthroAb_best_score": "best_score_full_argmax", "AnthroAb_masked": "masked_FR_only"}
REF_CMP  = {"AnthroAb_best_score": "Sapiens x AnthroAb_best_score",
            "AnthroAb_masked": "Sapiens x AnthroAb_masked_FRonly"}

for mode, key in MODE_KEY.items():
    sub = ref_cs[ref_cs.anthroab_mode == key]
    ref_pair = (len(sub), int(sub.identical_substitution_all3.sum()))
    my_pair = counts[mode]
    print(f"{key:24s} 레퍼런스 건드린 위치 {ref_pair[0]:2d} · 동일 치환 {ref_pair[1]:2d}   "
          f"내 결과 {my_pair[0]:2d} · {my_pair[1]:2d}  → {'일치' if ref_pair == my_pair else '차이'}")

display(ref_cs[ref_cs.identical_substitution_all3][["anthroab_mode", "chain", "imgt_position",
                                                    "sapiens_mut", "humatch_mut", "anthroab_mut"]])

# Sapiens × AnthroAb 겹침 — 내 표에서 직접 세고 레퍼런스와 맞춰요
# (overlap_summary.csv 의 note 컬럼은 6행 중 4행이 비어 있어 표에서 뺐어요)
ov_rows = []
for mode in ["AnthroAb_best_score", "AnthroAb_masked"]:
    for scope, sel in (("VH only", tbl.chain == "H"), ("VL only", tbl.chain == "L"),
                       ("VH+VL combined", tbl.chain.notna())):
        s = tbl[sel]
        ov_rows.append({"comparison": REF_CMP[mode], "chain_scope": scope,
                        "겹친 위치(내 결과)": len(set(s[s.tool == "Sapiens"].imgt)
                                              & set(s[s.tool == mode].imgt))})
ref_ov = pd.read_csv(REF / "overlap_summary.csv").rename(columns={"n_positions": "겹친 위치(레퍼런스)"})
ov = pd.DataFrame(ov_rows).merge(ref_ov[["comparison", "chain_scope", "겹친 위치(레퍼런스)"]],
                                 on=["comparison", "chain_scope"], how="left")
display(ov)
lo, hi = int(ov["겹친 위치(내 결과)"].min()), int(ov["겹친 위치(내 결과)"].max())
print(f"판정 — 같은 두 도구인데 겹침이 {lo}~{hi} 로 흔들려요. "
      "체인 범위와 AnthroAb 모드를 함께 적지 않은 겹침 숫자는 재현되지 않아요.")

## 이 랩에서 확인한 것

1. **Humatch** — config 가 `allow_CDR_mutations=False`·CNN 목표 **0.95**·GL 목표 **0.4** 를 선언하고, 산출물이 그대로 따라와요(CNN_H **0.972** · CNN_L **1.000** · gene **hv1/lv2** · edit **20** · CDR 변경 **0개**). VL 은 1 잔기 **삭제**(indel) 라 도구 간 비교는 IMGT 로.
2. **AnthroAb** — `predict_best_score` 는 CDR 을 **안 지켜요**. `predict_masked` 는 1.1.0 에서 깨져 있어(vocab 에 `*` 없음 → 토큰 밀림) `input_ids` 를 직접 만들어야 하고, 여기서는 본문 6.2.5 등급표를 무시한 **FR 전체 마스킹**으로 상한선을 봤어요.
3. **3도구 합의(실측)** — `best_score` 기준 건드린 위치 **7 · 동일 치환 7**, `masked` 기준 건드린 위치 **12 · 동일 치환 10**. 차이는 H45(`R→A` vs `R→P`)·H68(`N→A` vs `N→P`) 두 자리예요.
4. `I78T`(IMGT **H86**)는 `masked` 비교에서 세 도구가 모두 `I→T` 로 모인 자리이고, `best_score` 비교에서는 AnthroAb 가 이 자리를 아예 건드리지 않아 목록에 없어요. **합의 개수는 모드와 함께 적어야** 재현돼요.
5. 모델 덩치도 축이 달라요 — AnthroAb VH 는 **12층 · hidden 768 · 약 85M** 파라미터로, 같은 API 의 Sapiens(4층 · 128 · 약 0.5M) 와는 다른 체급이에요.

다음 → **[Ch.07 — Nativeness](../07_nativeness/07_nativeness_lab.ipynb)**